# PDF Retrieval And Workflows

This notebook indexes the sample PDFs, searches the resulting corpus, and runs a structured extraction workflow over retrieved evidence.


## Setup And Ingestion

The notebook is self-contained. It extracts the PDFs into an `AcademicDB` before indexing them.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import tempfile
from textwrap import shorten


def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "episcope").exists():
            return candidate
    return None


PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is not None:
    src_path = str(PROJECT_ROOT / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

pdf_candidates = [
    Path.cwd() / "pdf_samples",
    Path.cwd() / "notebooks" / "pdf_samples",
]
if PROJECT_ROOT is not None:
    pdf_candidates.append(PROJECT_ROOT / "notebooks" / "pdf_samples")

PDF_DIR = next((path for path in pdf_candidates if path.exists()), None)
if PDF_DIR is None:
    raise FileNotFoundError("Could not find the pdf_samples directory.")

pdf_paths = sorted(PDF_DIR.glob("*.pdf"))
if not pdf_paths:
    raise FileNotFoundError(f"No PDF files found in {PDF_DIR}.")

WORK_DIR = Path(tempfile.mkdtemp(prefix="episcope-pdf-"))
[path.name for path in pdf_paths], WORK_DIR


In [ ]:
from episcope.db.in_memory_academic_db import InMemoryAcademicDB
from episcope.rag.ingestion.document_loader import DocumentLoaderFactory

strategy_name = "pdf-samples-unstructured"
loader = DocumentLoaderFactory.get_loader("unstructured")
academic_db = InMemoryAcademicDB(backup_file=str(WORK_DIR / "academic_db.json"))


def persist_pdf(pdf_path: Path) -> dict[str, object]:
    sections, metadata, references = loader.load(pdf_path)
    metadata.file_path = str(pdf_path)

    doc_id = pdf_path.stem
    metadata_record = metadata.to_dict()
    metadata_record["original_paper_id"] = doc_id

    academic_db.insert(doc_id, "sections", strategy_name, [section.to_dict() for section in sections])
    academic_db.insert(doc_id, "metadata", strategy_name, metadata_record)
    academic_db.insert(doc_id, "references", strategy_name, [ref.to_dict() for ref in references])

    return {
        "doc_id": doc_id,
        "sections": len(sections),
        "characters": sum(len(section.content) for section in sections),
    }


loaded = [persist_pdf(path) for path in pdf_paths]
doc_ids = academic_db.list_docs(strategy_name)
loaded


## Build A PDF Index

The sample embedder is deterministic and offline, so rankings are only meant to exercise the API. Use a production embedder when ranking quality matters.


In [ ]:
import re
from typing import Iterable

import numpy as np

from episcope.rag.embeddings.base import Embedder


class TinyKeywordEmbedder(Embedder):
    vocabulary = (
        "data",
        "dataset",
        "source",
        "cohort",
        "survey",
        "registry",
        "trial",
        "patient",
        "hospital",
        "mortality",
        "covid",
        "treatment",
        "remdesivir",
        "supplement",
        "table",
        "figure",
        "reference",
        "database",
    )

    @property
    def model_name(self) -> str:
        return "tiny-keyword-demo"

    @property
    def dim(self) -> int:
        return len(self.vocabulary)

    def embed_text(self, text: str) -> list[float]:
        text = text.lower()
        counts = []
        for term in self.vocabulary:
            pattern = rf"\b{re.escape(term)}s?\b"
            counts.append(float(len(re.findall(pattern, text))))

        vector = np.array(counts, dtype="float32")
        norm = float(np.linalg.norm(vector))
        if norm:
            vector = vector / norm
        return vector.tolist()

    def embed_texts(self, texts: Iterable[str]) -> list[list[float]]:
        return [self.embed_text(text) for text in texts]


embedder = TinyKeywordEmbedder()
embedder.model_name, embedder.dim


In [ ]:
from episcope.rag.indexing.chunking import FixedSizeChunker
from episcope.rag.indexing.indexer import Indexer
from episcope.rag.retrieval.candidates import SemanticCandidateRetriever
from episcope.rag.retrieval.retriever import Retriever
from episcope.vectordb.file import FileDB

vdb = FileDB(str(WORK_DIR / "file_index"))
indexer = Indexer(
    vdb,
    embedder=embedder,
    chunker=FixedSizeChunker(chunk_size=1200, chunk_overlap=150),
)

for doc_id in doc_ids:
    sections = academic_db.retrieve(doc_id, "sections", strategy_name) or []
    metadata = academic_db.retrieve(doc_id, "metadata", strategy_name)
    indexer.index_paper(sections, metadata, paper_id=doc_id)

vdb.save()
semantic_candidates = SemanticCandidateRetriever(vdb, dense_embedder=embedder)
retriever = Retriever(vdb, candidate_retrievers=[semantic_candidates], use_rerank=False)

len(vdb.get_points()), vdb.get_embedding_model(), vdb.get_chunking_config()


## Search The PDF Corpus

Start with broad corpus retrieval to see what kinds of evidence the index returns.


In [ ]:
query = "data sources cohorts patient registry trial"
results = retriever.retrieve(query, top_k=5)

for rank, result in enumerate(results, start=1):
    preview = shorten(result.text.replace("\n", " "), width=220)
    print(f"{rank}. {result.paper_id} | score={result.similarity_score:.3f} | {preview}")


## Search Within One Paper

Paper-scoped retrieval is the default shape for extraction and classification workflows.


In [ ]:
paper_id = doc_ids[0]
paper_results = retriever.retrieve_by_paper(
    "patients treatment trial outcomes",
    paper_id,
    top_k=3,
)

for rank, result in enumerate(paper_results, start=1):
    preview = shorten(result.text.replace("\n", " "), width=220)
    print(f"{rank}. {result.paper_id} | score={result.similarity_score:.3f} | {preview}")


## Generate An Evidence Trace

Use `NoLLMGenerator` to verify the retrieval context and provenance before adding a model-backed generator.


In [ ]:
from episcope.rag.generation.nollm_generator import NoLLMGenerator

provenance = NoLLMGenerator().generate(results[:3], question=query)
print(shorten(provenance.answer.replace("\n", " "), width=1000))
[(evidence.paper_id, evidence.section) for evidence in provenance.evidences]


## Run A Structured Workflow

`PrecisionMiner` expects generator output that matches the extraction schema. This deterministic generator returns valid JSON so the workflow can be tested without API credentials.


In [ ]:
import json
from typing import Any, Callable, Optional, Sequence

from episcope.rag.generation.base import Generator
from episcope.rag.provenance import Evidence, Provenance
from episcope.workflows import PrecisionMiner
from episcope.workflows.precision_miner import FindDataSourcesConfig


class DemoDataSourceGenerator(Generator):
    model_id = "demo-json-generator"

    def generate(
        self,
        contexts: Sequence[Any],
        *,
        question: Optional[str] = None,
        message_builder: Optional[Callable[..., Any]] = None,
        **kwargs: Any,
    ) -> Provenance:
        contexts = list(contexts)
        items = []
        if contexts:
            first = contexts[0]
            items.append(
                {
                    "name": f"Retrieved evidence from {getattr(first, 'paper_id', '')}",
                    "url": None,
                    "explanation": "Example item produced from the highest-ranked retrieved chunk.",
                    "raw_text": shorten(getattr(first, "text", "").replace("\n", " "), width=500),
                }
            )

        payload = {
            "description": f"Demo extraction over {len(contexts)} retrieved chunks.",
            "items": items,
        }
        evidences = [
            Evidence(
                paper_id=getattr(ctx, "paper_id", ""),
                snippet=getattr(ctx, "text", ""),
                section=getattr(ctx, "section_type", None),
                model_id=self.model_id,
                prompt_id="demo-data-source-json",
            )
            for ctx in contexts
        ]
        return Provenance(answer=json.dumps(payload), evidences=evidences)


miner = PrecisionMiner(
    retriever=retriever,
    generator=DemoDataSourceGenerator(),
    strategy_name=strategy_name,
    config=FindDataSourcesConfig(top_k=4),
    academic_db=academic_db,
)

detailed = miner.run_detailed(doc_ids[0])
detailed.result.model_dump()


In [ ]:
print("Prompt messages:", len(detailed.trace.prompt_messages))
print("Retrieved chunks:", len(detailed.relevant_chunks))
print("Raw response:", detailed.trace.raw_llm_response)


The same retrieval and generator interfaces support model-backed extraction. Keep `run_detailed` in exploratory work because it exposes the prompt, retrieved chunks, raw response, parsed result, and provenance.
